# Comparing runs with 16 and 32 frames on WLASL 100 to 2000, with similar parameters

In [11]:
from typing import cast
import json
from pathlib import Path
#locals
# from code.run_types import ResSet, RunRes
import pandas as pd
from src.run_types import ResSet, RunRes, GenInfo
from src.resulting import print_json,  RESULTS_DIR, load_config_and_find_runs
from src.results.satnac_2026.filters import additional_modifications, exclude_keys

def load_find(conf_path: Path) -> GenInfo:
    runs =  load_config_and_find_runs(
            conf_path,
            exclude=exclude_keys,
            extra_mods=additional_modifications,
        )
    assert runs is not None
    return runs

In [12]:
results_dir = RESULTS_DIR / 'satnac_2026'

target_lengths = [16, 32]

runs_paths = {
    tl : results_dir / f'config_{tl}f_15p.toml'
    for tl in target_lengths
}

for runs_p in runs_paths.values():
    assert runs_p.exists(), f"{runs_p} not found"


## Parameters in common:

The only parameter that differs is the number of frames. In all cases, models trained on asl300 upward were initialised from the previous split. 

In [13]:
runs_by_tl = {
    tl: load_find(runs_p) 
    for tl, runs_p in runs_paths.items()
}

INFO resulting: Loaded que state from /home/luke/Code/SLR/src/que/Runs.json
INFO resulting: Found 34/188 runs matching the spec
INFO resulting: Excluded 24 runs based on additional modifications


INFO resulting: Loaded que state from /home/luke/Code/SLR/src/que/Runs.json
INFO resulting: Found 6/188 runs matching the spec
INFO resulting: Excluded 1 runs based on additional modifications


## Runs with different number of frames:

Lets check how many runs there are that match that spec:

In [14]:
for tl, runs in runs_by_tl.items():
    print(f'Target length: {tl}')
    print(f"Runs keys: {runs.keys()}")
    print(f"Number of entries: {len(runs['results'])}\n")

# print(json.dumps(runs_16['results'][0], indent=4))


Target length: 16
Runs keys: dict_keys(['spec', 'results'])
Number of entries: 10

Target length: 32
Runs keys: dict_keys(['spec', 'results'])
Number of entries: 5



## Now we can compare the runs

In [15]:
avail_acc_types = ["top_k_average_per_class_acc", "top_k_per_instance_acc"]
acc_type = avail_acc_types[1]
set_name = 'test'

#### Need to modify the names of S3D or they all get mixed together

In [16]:
df_format = []

for tl, runs in runs_by_tl.items():
    for res in runs["results"]:  
        model_name = res["admin"]["model"]
        df_format.append(
            {
                "model": model_name,
                "exp no": res['admin']['exp_no'],
                "run_id": res['wandb']['run_id'],
                "subset": res["admin"]["split"],
                "No. frames": tl
            }
            
            | {k: v for k, v in res["results"][set_name][acc_type].items()}
            | {"config path": res["admin"]["config_path"],
               "weight path": res['admin']['weight_path']}
        )    

df = pd.DataFrame(df_format) 

In [17]:
df = df.rename(columns={"top1": "Top-1", "top5": "Top-5", "top10": "Top-10"})
# df

In [18]:
df['Top-1'] = df['Top-1'].apply(lambda x: f'{x*100:.2f}')
df['Top-5'] = df['Top-5'].apply(lambda x: f'{x*100:.2f}')
df['Top-10'] = df['Top-10'].apply(lambda x: f'{x*100:.2f}')

In [19]:
subsets = ['asl100', 'asl300', 'asl1000', 'asl2000']
for set_name in subsets:
    print(f'{set_name}'.capitalize())
    subdf = df[df['subset'] == set_name]
    display(subdf.sort_values('Top-1', ascending=False))

Asl100


,model,exp no,run_id,subset,No. frames,Top-1,Top-5,Top-10,config path,weight path
9,MViTv2_S,007,None,asl100,16,79.46,91.86,95.35,configfiles/generic/lframe_hwd_warmrestarts.toml,None
8,MViTv2_S,010,xuvgm50f,asl100,16,73.26,91.47,94.57,configfiles/asl100/MViTv2_S/exp010.toml,None
13,S3D,035,r22x5871,asl100,32,63.18,86.05,91.09,configfiles/generic/wups_restarts_et0.toml,None
14,S3D,023,yaylbndm,asl100,32,58.14,86.43,92.64,configfiles/generic/hframe_hwd_warmrestarts.toml,None
0,S3D,051,f6r2dos3,asl100,16,53.10,79.84,87.98,configfiles/asl100/S3D/exp046.toml,None
7,S3D,046,5xgru0cs,asl100,16,50.39,77.91,86.43,configfiles/asl100/S3D/exp046.toml,None


Asl300


,model,exp no,run_id,subset,No. frames,Top-1,Top-5,Top-10,config path,weight path
5,MViTv2_S,001,nfwehytd,asl300,16,65.42,88.92,93.56,configfiles/asl300/MViTv2_S/exp001.toml,None
12,S3D,046,None,asl300,32,54.49,80.69,87.28,configfiles/generic/hframe_hwd_warmrestarts.toml,None
6,S3D,002,vty80mdi,asl300,16,51.95,78.44,87.13,configfiles/asl300/S3D/exp002.toml,None


Asl1000


,model,exp no,run_id,subset,No. frames,Top-1,Top-5,Top-10,config path,weight path
3,MViTv2_S,000,rc5m3meh,asl1000,16,53.41,81.77,87.79,configfiles/asl1000/MViTv2_S/exp000.toml,None
11,S3D,047,na70ufkk,asl1000,32,37.74,67.86,77.72,configfiles/generic/hframe_hwd_warmrestarts.toml,None
4,S3D,000,kpdu2hra,asl1000,16,36.25,67.80,77.08,configfiles/asl1000/S3D/exp000.toml,None


Asl2000


,model,exp no,run_id,subset,No. frames,Top-1,Top-5,Top-10,config path,weight path
2,MViTv2_S,000,fkv6kpik,asl2000,16,38.87,71.62,79.82,configfiles/asl2000/MViTv2_S/exp000.toml,None
1,S3D,000,olp97b32,asl2000,16,19.62,46.96,58.25,configfiles/asl2000/S3D/exp000.toml,None
10,S3D,048,None,asl2000,32,17.12,41.02,53.28,configfiles/generic/hframe_hwd_warmrestarts.toml,None
